In [1]:
import ibis
from ibis import _, selectors as s
from utils.f_0_dirs import get_data_dirs
dirs = get_data_dirs(segment="descriptives")
con = ibis.duckdb.connect(dirs.db_path, read_only=True)

t_distances = con.table("working_distance_ttwa_km")
t_fixed = con.table("working_fixed")

In [38]:
import numpy as np

# Returns a LaTeX bmatrix
# :a: numpy array
# :decimals: integer, number of decimal places to round to
# :returns: LaTeX bmatrix as a string
def bmatrix(a: np.ndarray, decimals: int = 2) -> str:
    if len(a.shape) > 2:
        raise ValueError('bmatrix can at most display two dimensions')
        
    # Handle 1D arrays by promoting them to 2D row vectors
    if len(a.shape) == 1:
        a = a.reshape(1, -1)

    rv = [r'\begin{bmatrix}']
    
    for row in a:
        # Format numbers to 3 decimal places for a cleaner table[cite: 4]
        formatted_row = [f"{float(val):.{decimals}f}" if float(val) != 0 else r"\footnotesize{0}" for val in row]
        rv.append('    ' + ' & '.join(formatted_row) + r' \\')
        
    rv.append(r'\end{bmatrix}')
    
    return '\n'.join(rv)

# Example Usage:
my_array = np.random.rand(3, 3)
print(bmatrix(my_array))

\begin{bmatrix}
    0.14 & 0.40 & 0.44 \\
    0.03 & 0.22 & 0.27 \\
    0.47 & 0.12 & 0.11 \\
\end{bmatrix}


In [39]:
import pandas as pd
import numpy as np
from scipy.linalg import block_diag

# Pick 2 random firms from t_distances
random_ids = (
    t_distances
    .limit(1000)
    .order_by(ibis.random())
    .distinct(on='firm_i')
    .limit(2)
    .execute()
)
ids = random_ids['firm_i'].tolist()

chain_lengths = [3, 4]
chains = []
for index, firm_i in enumerate(ids):
    ch = chain_lengths[index]

    # Find an id that is 7 spatial steps away from firm_i
    curr_chain = [firm_i]
    curr_id = firm_i
    while len(curr_chain) <= ch:
        firm_j = (
            t_distances
            .filter((_.firm_i == curr_id) & (_.distance_meters > 0))
            .order_by(_.distance_meters.asc())
            .limit(1)
        )
        curr_id = firm_j.execute()['firm_j'].iloc[0]
        curr_chain.append(curr_id)
    t_distance_filtered = (
        t_distances
        .filter(
            (_.firm_i.isin(curr_chain)) &
            (_.firm_j.isin(curr_chain))
        )
        .mutate(
            d1 = 1 / (_.distance_meters + 1),
            d2 = 1 / (_.distance_meters + 1) ** 2,
            d3 = (-_.distance_meters / 1000).exp()
        )
        # 1. Normalize vertically BEFORE pivoting using a Window function
        .mutate(
            d1 = _.d1 / _.d1.sum().over(group_by=_.firm_i)
        )
        .select('firm_i', 'firm_j', 'd1')
        .pivot_wider(
            names_from='firm_j',
            values_from='d1',
            values_fill=0
        )
        .execute()
        .set_index('firm_i')
        # 2. Columns and indices sorted by the ordered chain
        .reindex(columns=curr_chain, index=curr_chain)
    )
    chains.append(t_distance_filtered)

np_combined = block_diag(chains[0].to_numpy(), chains[1].to_numpy())
print(bmatrix(np_combined))

\begin{bmatrix}
    \footnotesize{0} & 1.00 & \footnotesize{0} & 1.00 & \footnotesize{0} & \footnotesize{0} & \footnotesize{0} & \footnotesize{0} & \footnotesize{0} \\
    1.00 & \footnotesize{0} & 1.00 & \footnotesize{0} & \footnotesize{0} & \footnotesize{0} & \footnotesize{0} & \footnotesize{0} & \footnotesize{0} \\
    \footnotesize{0} & 1.00 & \footnotesize{0} & 1.00 & \footnotesize{0} & \footnotesize{0} & \footnotesize{0} & \footnotesize{0} & \footnotesize{0} \\
    1.00 & \footnotesize{0} & 1.00 & \footnotesize{0} & \footnotesize{0} & \footnotesize{0} & \footnotesize{0} & \footnotesize{0} & \footnotesize{0} \\
    \footnotesize{0} & \footnotesize{0} & \footnotesize{0} & \footnotesize{0} & \footnotesize{0} & 0.60 & 0.40 & 0.60 & 0.40 \\
    \footnotesize{0} & \footnotesize{0} & \footnotesize{0} & \footnotesize{0} & 0.36 & \footnotesize{0} & 0.64 & \footnotesize{0} & 0.64 \\
    \footnotesize{0} & \footnotesize{0} & \footnotesize{0} & \footnotesize{0} & 0.27 & 0.73 & \footnotesize{